# Dropout and Maxout

In this lab, you will explore two important techniques:
1. **Dropout**: A regularisation method that randomly drops activations during training
2. **Maxout**: A non-linear transformation for multi-layer models

These methods are based on material from Lecture 6. For more details, see:
- [Dropout paper](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf)
- [Maxout paper](https://arxiv.org/pdf/1302.4389.pdf)

## Exercise 1: Implementing a Dropout Layer

### Background

During training, dropout randomly sets a subset of input dimensions to zero. Each dimension has a probability $p$ of being included (not dropped), with all dimensions sampled independently.

We can represent dropout as an elementwise multiplication with a **binary mask** vector $\boldsymbol{m} = \left[m_1 ~ m_2 ~\dots~ m_D\right]^{\rm T}$, where $m_d \sim \text{Bernoulli}(p)$ for all dimensions $d$.

### Task 1.1: Implement `random_binary_mask`

Complete the function below to generate a binary mask array where:
- Each value is either 0 or 1
- Probability of 1 is `prob_1`
- All values are sampled independently

In [9]:
import numpy as np

def random_binary_mask(prob_1, shape, rng):
    """Generates a random binary mask array of a given shape.
    
    Each value in the output array is independently sampled as a binary 
    value (0 or 1), with probability prob_1 of being 1.
    
    Args:
        prob_1: Scalar in [0, 1] specifying probability of each entry being 1.
        shape: Shape of the returned mask array.
        rng (RandomState): Seeded random number generator object.
    
    Returns:
        Random binary mask array of specified shape.
    """
    # STUDENT: Complete this function
    return np.random.binomial(1, prob_1, shape)

Run the test cell below to verify your `random_binary_mask` implementation. If incorrect, you'll get an `AssertionError` indicating what failed.

In [10]:
import numpy as np

test_shapes = [(1, 1000), (10, 10, 10)]
test_probs = [0.1, 0.5, 0.7]

for i in range(10):
    for shape in test_shapes:
        for prob in test_probs:
            output = random_binary_mask(prob, shape, np.random)
            
            # Check correct shape
            assert output.shape == shape, f"Expected shape {shape}, got {output.shape}"
            
            # Check all outputs are binary (0 or 1)
            assert np.all((output == 1.) | (output == 0.)), "Output contains non-binary values"
            
            # Check proportion of 1s is close to prob (allowing for randomness)
            assert np.abs(output.mean() - prob) < 0.1, f"Mean {output.mean():.3f} not close to {prob}"

print("All tests passed! ✓")

All tests passed! ✓


### Task 1.2: Implement Dropout Forward and Backward Propagation

#### Forward Propagation

Given a binary mask $\boldsymbol{m}$, the stochastic forward pass through dropout is:

$$y^{(b)}_d = m_d \cdot x^{(b)}_d \qquad \forall d \in \lbrace 1 \dots D \rbrace$$

#### Backward Propagation

The partial derivatives for backpropagation are:

$$\frac{\partial y^{(b)}_k}{\partial x^{(b)}_d} = 
\begin{cases}
    m_k & \text{if } k = d \\
    0   & \text{if } k \neq d
\end{cases}$$

#### Test Time Behavior

At test time, dropout is **not** applied stochastically. Instead, we scale inputs by $p$ (the inclusion probability) to maintain expected activation levels:

$$z^{(b)}_d = p \cdot x^{(b)}_d$$

This ensures the expected output at test time matches the expected output during training.

### Implementation

The `StochasticLayer` class (in `mlp.layers`) has an `fprop` method with a `stochastic` parameter:
- `stochastic=True` (default): Apply stochastic dropout during training
- `stochastic=False`: Apply deterministic scaling at test time

**Instructions:** Complete the `fprop` and `bprop` methods in the `DropoutLayer` class below. Store the mask as `self._mask` in `fprop` to use it in `bprop`.

In [15]:
from mlp.layers import StochasticLayer

class DropoutLayer(StochasticLayer):
    """Layer which stochastically drops input dimensions in its output."""
    
    def __init__(self, rng=None, incl_prob=0.5, share_across_batch=True):
        """Construct a new dropout layer.
        
        Args:
            rng (RandomState): Seeded random number generator.
            incl_prob: Scalar in (0, 1] specifying probability of each input 
                dimension being included (not dropped) in output.
            share_across_batch: Whether to use same dropout mask across all 
                inputs in a batch (True) or use per-input masks (False).
        """
        super(DropoutLayer, self).__init__(rng)
        assert incl_prob > 0. and incl_prob <= 1., "incl_prob must be in (0, 1]"
        self.incl_prob = incl_prob
        self.share_across_batch = share_across_batch
        
    def fprop(self, inputs, stochastic=True):
        """Forward propagates activations through the layer.

        Args:
            inputs: Array of shape (batch_size, input_dim).
            stochastic: If True, apply stochastic dropout (training mode).
                If False, apply deterministic scaling (test mode).

        Returns:
            outputs: Array of shape (batch_size, output_dim).
        """
        if stochastic:
            if self.share_across_batch:
                shape = (1, inputs.shape[1])
                self._mask = np.tile(random_binary_mask(self.incl_prob, shape, self.rng), (inputs.shape[0], 1))
                return inputs * self._mask 
            else:
                self._mask = random_binary_mask(self.incl_prob, inputs.shape, self.rng)
                return inputs * self._mask
        else: 
            return self.incl_prob * inputs 
    
    def bprop(self, inputs, outputs, grads_wrt_outputs):
        """Back propagates gradients through the layer.

        Args:
            inputs: Array of shape (batch_size, input_dim).
            outputs: Array of shape (batch_size, output_dim).
            grads_wrt_outputs: Array of shape (batch_size, output_dim).

        Returns:
            grads_wrt_inputs: Array of shape (batch_size, input_dim).
        """
  
        return self._mask * grads_wrt_outputs


    def __repr__(self):
        return f'DropoutLayer(incl_prob={self.incl_prob:.1f})'

Test your dropout implementation using the cell below. If incorrect, you'll get an `AssertionError` with hints about what's wrong.

In [16]:
seed = 31102016 
rng = np.random.RandomState(seed)
test_incl_probs = [0.1, 0.5, 0.7]
input_shape = (5, 10)

for incl_prob in test_incl_probs:
    layer = DropoutLayer(rng, incl_prob)
    inputs = rng.normal(size=input_shape)
    grads_wrt_outputs = rng.normal(size=input_shape)
    
    # Test stochastic forward pass multiple times
    for t in range(100):
        outputs = layer.fprop(inputs, stochastic=True)
        
        # Check correct shape
        assert outputs.shape == inputs.shape, "Output shape mismatch"
        
        # Check outputs are either equal to inputs or zero
        assert np.all((outputs == inputs) | (outputs == 0)), "Invalid dropout behavior"
        
        # Test backpropagation
        grads_wrt_inputs = layer.bprop(inputs, outputs, grads_wrt_outputs)
        
        # Check gradients only non-zero where outputs are non-zero
        assert np.all((outputs != 0) == (grads_wrt_inputs != 0)), "Gradient masking error"
        assert np.all(grads_wrt_outputs[outputs != 0] == grads_wrt_inputs[outputs != 0]), \
            "Gradient values incorrect"
    
    # Test deterministic forward pass (test time)
    det_outputs = layer.fprop(inputs, stochastic=False)
    assert det_outputs.shape == inputs.shape, "Deterministic output shape mismatch"
    assert np.allclose(det_outputs, incl_prob * inputs), "Deterministic scaling incorrect"

print("All dropout tests passed! ✓")

All dropout tests passed! ✓


### Optional Extension

The implementation above uses the same dropout mask for all inputs in a batch (`share_across_batch=True`). In practice, using different masks per input can sometimes improve performance. 

**Optional:** Modify the `DropoutLayer` to support per-input dropout masks by using the `share_across_batch` parameter.

## Exercise 2: Training with Dropout

### Task 2: Experiment with Dropout Regularization

Now train models with dropout layers to classify MNIST digits. The code below provides a starting point.

**Your experiments should explore:**

1. **Different `incl_prob` values**: Try 0.3, 0.5, 0.7, 0.9
2. **Layer-specific dropout**: Different probabilities for input vs hidden layers
3. **Model architecture**: Larger models with more parameters benefit more from dropout
4. **Training epochs**: Dropout models typically need more epochs (why?)

**Questions to consider:**
- How does dropout affect training vs validation error?
- What happens with very low `incl_prob` (e.g., 0.2)?
- Why might you use different dropout rates at input vs hidden layers?

You may want to start Exercise 3 while waiting for training runs to complete.

In [ ]:
import numpy as np
import logging
from mlp.data_providers import MNISTDataProvider
from mlp.models import MultipleLayerModel
from mlp.layers import ReluLayer, AffineLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.initialisers import GlorotUniformInit, ConstantInit
from mlp.learning_rules import MomentumLearningRule
from mlp.optimisers import Optimiser
import matplotlib.pyplot as plt
%matplotlib inline

# Seed random number generator for reproducibility
seed = 31102016 
rng = np.random.RandomState(seed)

# Set up logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers = [logging.StreamHandler()]

# Create data providers
train_data = MNISTDataProvider('train', batch_size=50, rng=rng)
valid_data = MNISTDataProvider('valid', batch_size=50, rng=rng)

In [ ]:
# Hyperparameters
incl_prob = 0.5  # STUDENT: Try different values (0.3, 0.5, 0.7, 0.9)
input_dim, output_dim, hidden_dim = 784, 10, 125

# Initialize weights and biases
weights_init = GlorotUniformInit(rng=rng, gain=2.**0.5)
biases_init = ConstantInit(0.)

# Define model: 3 affine layers with ReLU activations and dropout
model = MultipleLayerModel([
    DropoutLayer(rng, incl_prob),
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), 
    ReluLayer(),
    DropoutLayer(rng, incl_prob),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), 
    ReluLayer(),
    DropoutLayer(rng, incl_prob),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init)
])

# Loss function for multi-class classification
error = CrossEntropySoftmaxError()

# Optimizer with momentum
learning_rule = MomentumLearningRule(learning_rate=0.02, mom_coeff=0.9)

# Monitor accuracy during training
data_monitors = {'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}

# Create optimizer
optimiser = Optimiser(
    model, error, learning_rule, train_data, valid_data, data_monitors)

# Training parameters
num_epochs = 100
stats_interval = 5

# Train model
stats, keys, run_time = optimiser.train(
    num_epochs=num_epochs, stats_interval=stats_interval)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Error plot
for k in ['error(train)', 'error(valid)']:
    ax1.plot(np.arange(1, stats.shape[0]) * stats_interval, 
             stats[1:, keys[k]], label=k)
ax1.legend()
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Error')
ax1.set_title('Training and Validation Error')

# Accuracy plot
for k in ['acc(train)', 'acc(valid)']:
    ax2.plot(np.arange(1, stats.shape[0]) * stats_interval, 
             stats[1:, keys[k]], label=k)
ax2.legend()
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')

plt.tight_layout()
plt.show()

print(f"\nTraining completed in {run_time:.2f} seconds")
print(f"Final validation accuracy: {stats[-1, keys['acc(valid)']]:.4f}")

## Exercise 3: Implementing Maxout

### Background

[Maxout](https://arxiv.org/pdf/1302.4389.pdf) is a generalization of the ReLU activation function.

**ReLU** takes the maximum of input and zero:
$$y^{(b)}_k = \max\lbrace 0,\, x^{(b)}_k \rbrace$$

**Maxout** takes the maximum over groups of inputs of size $s$:
$$y^{(b)}_k = \max\lbrace x^{(b)}_{(k-1)s + 1},\, x^{(b)}_{(k-1)s + 2},\, \dots,\, x^{(b)}_{ks} \rbrace$$

When these inputs come from an affine layer, maxout learns piecewise linear activations without saturating gradients.

### Properties

- **Piecewise linear**: Like ReLU, but more flexible
- **No forced zeros**: Unlike ReLU, all inputs can contribute
- **Good empirical performance**: Especially with dropout
- **Used in CNNs**: Similar to max-pooling (covered in later labs)

### Gradients

The gradient is sparse - only the maximum input in each pool has non-zero gradient:

$$\frac{\partial y^{(b)}_k}{\partial x^{(b)}_d} = 
\begin{cases} 
  1 & \text{if } d = \arg\max_{i \in \text{pool}_k} x^{(b)}_i \\
  0 & \text{otherwise}
\end{cases}$$

### Task 3: Implement Max-Pooling Layer

Complete the `fprop` and `bprop` methods below.

**Hints:**
- Use `numpy.reshape` to organize inputs into non-overlapping pools
- Use `numpy.max` with the `axis` parameter to take max over pools
- Create a binary mask in `fprop` to identify maximum values for use in `bprop`
- Store the mask as `self._mask` for use in backpropagation

In [ ]:
from mlp.layers import Layer

class MaxPoolingLayer(Layer):
    """Layer that takes maximum over non-overlapping pools of inputs."""
    
    def __init__(self, pool_size=2):
        """Construct a new max-pooling layer.
        
        Args:
            pool_size: Positive integer specifying pool size. The input
                dimension must be a multiple of pool_size.
        """
        assert pool_size > 0, "pool_size must be positive"
        self.pool_size = pool_size

    def fprop(self, inputs):
        """Forward propagates activations through the layer.
        
        Takes maximum over non-overlapping pools of size pool_size.

        Args:
            inputs: Array of shape (batch_size, input_dim).

        Returns:
            outputs: Array of shape (batch_size, input_dim // pool_size).
        """
        # STUDENT: Complete this method
        assert inputs.shape[-1] % self.pool_size == 0, (
            f'Input dimension {inputs.shape[-1]} must be multiple of pool_size {self.pool_size}')
        
        raise NotImplementedError("TODO implement this")
    
    
    def bprop(self, inputs, outputs, grads_wrt_outputs):
        """Back propagates gradients through the layer.

        Args:
            inputs: Array of shape (batch_size, input_dim).
            outputs: Array of shape (batch_size, output_dim).
            grads_wrt_outputs: Array of shape (batch_size, output_dim).

        Returns:
            grads_wrt_inputs: Array of shape (batch_size, input_dim).
        """
        raise NotImplementedError("TODO implement this")

    def __repr__(self):
        return f'MaxPoolingLayer(pool_size={self.pool_size})'

Test your max-pooling implementation using the cell below.

In [ ]:
# Test data
test_inputs = np.array([[-3, -4, 5, 8], [0, -2, 3, -8], [1, 5, 3, 2]])

# Expected outputs and gradients for pool_size=4
test_outputs_1 = np.array([[8], [3], [5]])
test_grads_wrt_outputs_1 = np.array([[10], [5], [-3]])
test_grads_wrt_inputs_1 = np.array([[0, 0, 0, 10], [0, 0, 5, 0], [0, -3, 0, 0]])

# Expected outputs and gradients for pool_size=2
test_outputs_2 = np.array([[-3, 8], [0, 3], [5, 3]])
test_grads_wrt_outputs_2 = np.array([[3, -1], [2, 5], [5, 3]])
test_grads_wrt_inputs_2 = np.array([[3, 0, 0, -1], [2, 0, 5, 0], [0, 5, 3, 0]])

# Test with pool_size=4
layer_1 = MaxPoolingLayer(4)
outputs_1 = layer_1.fprop(test_inputs)
assert np.allclose(outputs_1, test_outputs_1), "Forward pass failed for pool_size=4"

grads_1 = layer_1.bprop(test_inputs, test_outputs_1, test_grads_wrt_outputs_1)
assert np.allclose(grads_1, test_grads_wrt_inputs_1), "Backward pass failed for pool_size=4"

# Test with pool_size=2
layer_2 = MaxPoolingLayer(2)
outputs_2 = layer_2.fprop(test_inputs)
assert np.allclose(outputs_2, test_outputs_2), "Forward pass failed for pool_size=2"

grads_2 = layer_2.bprop(test_inputs, test_outputs_2, test_grads_wrt_outputs_2)
assert np.allclose(grads_2, test_grads_wrt_inputs_2), "Backward pass failed for pool_size=2"

print("All max-pooling tests passed! ✓")

## Exercise 4: Training with Maxout

### Task 4: Experiment with Maxout Networks

Use your `MaxPoolingLayer` to build and train models for MNIST digit classification.

**Experiments to try:**

1. **Different pool sizes**: Try `pool_size` = 2, 3, 4
2. **Combine with dropout**: Does maxout + dropout work better?
3. **Compare to ReLU**: Train equivalent models with ReLU vs maxout
4. **Vary architecture**: Try different numbers of layers and hidden dimensions

**Note:** When using maxout, the affine layer before it should output `hidden_dim * pool_size` dimensions, so after pooling you get `hidden_dim` dimensions.

In [ ]:
import numpy as np
import logging
from mlp.data_providers import MNISTDataProvider
from mlp.models import MultipleLayerModel
from mlp.layers import AffineLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.initialisers import GlorotUniformInit, ConstantInit
from mlp.learning_rules import MomentumLearningRule
from mlp.optimisers import Optimiser
import matplotlib.pyplot as plt
%matplotlib inline

# Seed random number generator
seed = 31102016 
rng = np.random.RandomState(seed)

# Set up logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers = [logging.StreamHandler()]

# Create data providers
train_data = MNISTDataProvider('train', batch_size=50, rng=rng)
valid_data = MNISTDataProvider('valid', batch_size=50, rng=rng)

In [ ]:
# Hyperparameters
pool_size = 2  # STUDENT: Try different values (2, 3, 4)
input_dim, output_dim, hidden_dim = 784, 10, 100

# Initialize weights and biases
weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

# Define model: affine layers interleaved with max-pooling
# Note: Affine layers output hidden_dim * pool_size, then pooling reduces to hidden_dim
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim * pool_size, weights_init, biases_init), 
    MaxPoolingLayer(pool_size),
    AffineLayer(hidden_dim, hidden_dim * pool_size, weights_init, biases_init), 
    MaxPoolingLayer(pool_size),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init)
])

# Loss function
error = CrossEntropySoftmaxError()

# Optimizer
learning_rule = MomentumLearningRule(learning_rate=0.02, mom_coeff=0.9)

# Monitor accuracy
data_monitors = {'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}

# Create optimizer
optimiser = Optimiser(
    model, error, learning_rule, train_data, valid_data, data_monitors)

# Training parameters
num_epochs = 100
stats_interval = 5

# Train model
stats, keys, run_time = optimiser.train(
    num_epochs=num_epochs, stats_interval=stats_interval)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Error plot
for k in ['error(train)', 'error(valid)']:
    ax1.plot(np.arange(1, stats.shape[0]) * stats_interval, 
             stats[1:, keys[k]], label=k)
ax1.legend()
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Error')
ax1.set_title('Training and Validation Error')

# Accuracy plot
for k in ['acc(train)', 'acc(valid)']:
    ax2.plot(np.arange(1, stats.shape[0]) * stats_interval, 
             stats[1:, keys[k]], label=k)
ax2.legend()
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')

plt.tight_layout()
plt.show()

print(f"\nTraining completed in {run_time:.2f} seconds")
print(f"Final validation accuracy: {stats[-1, keys['acc(valid)']]:.4f}")

---

# PyTorch Implementation

In this section, you'll implement dropout and maxout using PyTorch's built-in functionality.

## Background: Dropout in PyTorch

Dropout is a regularization technique that:
- **Randomly drops neurons** during training (sets their activations to 0)
- **Prevents overfitting** by forcing the network to learn robust features
- **Acts like ensemble learning** where each training step uses a different sub-network

PyTorch provides a [`nn.Dropout`](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html) module that handles both training and test modes automatically.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data.sampler import SubsetRandomSampler

torch.manual_seed(seed)

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
batch_size = 128
learning_rate = 0.001
num_epochs = 100
stats_interval = 5
prob = 0.5  # Dropout probability (probability of zeroing)

# Model dimensions
input_dim = 1 * 28 * 28  # Grayscale 28x28 images
output_dim = 10  # 10 digit classes
hidden_dim = 125

In [ ]:
# Data transformations (normalize with MNIST mean and std)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST datasets
train_dataset = datasets.MNIST('../data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('../data', train=False, download=True, transform=transform)

# Create train/validation split (80/20)
valid_size = 0.2
num_train = len(train_dataset)
indices = list(range(num_train))
split = int(np.floor(valid_size * num_train))
np.random.shuffle(indices)
train_idx, valid_idx = indices[split:], indices[:split]

# Create samplers
train_sampler = SubsetRandomSampler(train_idx)
valid_sampler = SubsetRandomSampler(valid_idx)

# Create data loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, sampler=train_sampler, pin_memory=True)
valid_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, sampler=valid_sampler, pin_memory=True)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

print(f"Training samples: {len(train_idx)}")
print(f"Validation samples: {len(valid_idx)}")
print(f"Test samples: {len(test_dataset)}")

## PyTorch Dropout Model

Implementing dropout in PyTorch is straightforward using the `nn.Dropout` module:
- Takes a single parameter `p`: probability of an element being zeroed
- Automatically switches behavior between training and evaluation modes
- During training: randomly zeros elements and scales remaining by 1/(1-p)
- During evaluation: acts as identity (no dropout)

In [ ]:
class DropoutMultipleLayerModel(nn.Module):
    """Multi-layer neural network with dropout."""
    
    def __init__(self, input_dim, output_dim, hidden_dim):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Dropout(p=prob),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Dropout(p=prob),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Dropout(p=prob)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# Create model    
model = DropoutMultipleLayerModel(input_dim, output_dim, hidden_dim).to(device)

# Loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(model)

In [ ]:
# Training loop
train_losses, valid_losses = [], []
train_accs, valid_accs = [], []

for epoch in range(num_epochs + 1):
    # Training phase
    model.train()  # Enable dropout
    batch_loss, batch_acc = [], []
    
    for x, t in train_loader:
        x, t = x.to(device), t.to(device)
        
        # Forward pass
        y = model(x)
        loss = loss_fn(y, t)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        acc = (t == y.argmax(1)).float().mean()
        batch_loss.append(loss.item())
        batch_acc.append(acc.item())
    
    train_losses.append(np.mean(batch_loss))
    train_accs.append(np.mean(batch_acc))

    # Validation phase
    model.eval()  # Disable dropout
    batch_loss, batch_acc = [], []
    
    with torch.no_grad():  # No gradients needed
        for x, t in valid_loader:
            x, t = x.to(device), t.to(device)
            
            # Forward pass
            y = model(x)
            loss = loss_fn(y, t)
            
            # Calculate accuracy
            acc = (t == y.argmax(1)).float().mean()
            batch_loss.append(loss.item())
            batch_acc.append(acc.item())
    
    valid_losses.append(np.mean(batch_loss))
    valid_accs.append(np.mean(batch_acc))

    # Print statistics
    if epoch % stats_interval == 0:
        print(f'Epoch {epoch:3d}: '
              f'Train Loss={train_losses[-1]:.4f}, Train Acc={train_accs[-1]:.4f} | '
              f'Valid Loss={valid_losses[-1]:.4f}, Valid Acc={valid_accs[-1]:.4f}')

print("\nTraining complete!")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(train_losses, label='Train')
ax1.plot(valid_losses, label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(train_accs, label='Train')
ax2.plot(valid_accs, label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final validation accuracy: {valid_accs[-1]:.4f}")

## PyTorch Maxout

In PyTorch, maxout can be implemented using [`nn.MaxPool1d`](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool1d.html) or by manually reshaping and taking maximums.

We'll explore maxout and convolutional layers (which also use max-pooling) in detail in the next lab on Convolutional Neural Networks.

**Note:** The relationship between maxout and max-pooling will become clearer when we cover CNNs, where max-pooling over spatial dimensions is a standard technique.